# Role-Aware SAAMR: SDF to OpenFF/OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook starts from SDF files generated by `Role_Aware_SAAMR_Quickstart.ipynb` and demonstrates the downstream handoff:

```text
Primitive -> RDKit -> SDF -> OpenFF -> OpenMM
```

Generated simulation artifacts are written under `examples_system/role_aware_saamr_outputs/` and are ignored by git.

## 1. Environment and Inputs

In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

SDF_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "sdf"
SIM_DIR = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "openmm"
SIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"SDF directory: {SDF_DIR}")
print(f"Simulation output directory: {SIM_DIR}")

## 2. Load SDF Files with RDKit

In [ ]:
from rdkit import Chem

sdf_paths = sorted(SDF_DIR.glob("*.sdf"))
if not sdf_paths:
    raise FileNotFoundError(
        f"No SDF files found in {SDF_DIR}. Run Role_Aware_SAAMR_Quickstart.ipynb first."
    )

rdkit_mols = []
rdkit_mol_sources = []
for path in sdf_paths:
    supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
    for record_idx, mol in enumerate(supplier):
        if mol is None:
            raise ValueError(f"Could not read record {record_idx} from {path}")
        Chem.SanitizeMol(Chem.Mol(mol))
        rdkit_mols.append(mol)
        rdkit_mol_sources.append((path, record_idx))

print(f"Loaded {len(rdkit_mols)} SDF molecule(s)")
for (path, record_idx), mol in zip(rdkit_mol_sources, rdkit_mols):
    print(f"  {path.name}[{record_idx}]: atoms={mol.GetNumAtoms()}, bonds={mol.GetNumBonds()}")

## 3. Validate MuPT SAAMR Metadata

This streams through MuPT SDF metadata to confirm the files retain role-aware hierarchy information without reconstructing every Primitive segment.

In [ ]:
from mupt.interfaces.rdkit import summarize_mupt_sdf

sdf_summary = summarize_mupt_sdf(sdf_paths)

assert sdf_summary["records"] == len(rdkit_mols)
print(
    f"validated {sdf_summary['records']} SDF record(s): "
    f"residues={sdf_summary['residues']}, atoms={sdf_summary['atoms']}"
)
print(f"segment labels: {sdf_summary['segment_label_counts']}")

## 4. OpenFF/OpenMM Availability and Run Settings

The workflow below treats the SDF random-walk coordinates as intentional vacuum starting conformations: each molecule is charged with an OpenFF graph neural network model, parameterized, minimized, and briefly evolved in vacuum so the initially linear polymers can collapse without PME overhead. After that collapse, the notebook can wrap a periodic box around the relaxed coordinates for optional NPT coalescence.

The charge model is `openff-gnn-am1bcc-1.0.0.pt`, avoiding per-molecule AM1-BCC calculations. This model is provided by OpenFF NAGL, not RDKit, AmberTools, or the built-in OpenFF toolkit wrappers. If NAGL is unavailable, the parameterization cell skips with installation instructions rather than falling back to AM1-BCC.

This is not a production melt-equilibration protocol. It is a pragmatic tutorial path that avoids huge sparse PME boxes while giving energy minimization and short MD a non-overlapping starting structure.

In [ ]:
try:
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.interchange import Interchange
    from openff.toolkit.utils import ToolkitRegistry
    from openff.units import unit as off_unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

NAGL_AVAILABLE = False
NAGL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

        # Importing the wrapper is not enough: NAGLToolkitWrapper can be present
        # in OpenFF Toolkit even when the openff-nagl package is not installed.
        NAGL_AVAILABLE = NAGLToolkitWrapper.is_available()
        if not NAGL_AVAILABLE:
            NAGL_IMPORT_ERROR = RuntimeError(
                "OpenFF Toolkit provides NAGLToolkitWrapper, but the OpenFF NAGL "
                "backend is unavailable. Install openff-nagl. See "
                "https://docs.openforcefield.org/projects/nagl/en/latest/installation.html"
            )
    except ModuleNotFoundError as exc:
        NAGL_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
    from openmm import unit as omm_unit
    from openmm.app import DCDReporter, PDBFile, StateDataReporter
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"

RUN_OPENFF_PARAMETERIZATION = OPENFF_AVAILABLE and NAGL_AVAILABLE
RUN_VACUUM_COLLAPSE = RUN_OPENFF_PARAMETERIZATION and OPENMM_AVAILABLE
RUN_PERIODIC_NPT = False  # Set True after inspecting the vacuum-collapse result.
VACUUM_COLLAPSE_STEPS = 500
PERIODIC_NPT_STEPS = 250
PERIODIC_PADDING_NM = 3.0

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenFF NAGL available: {NAGL_AVAILABLE}")
if not NAGL_AVAILABLE and NAGL_IMPORT_ERROR is not None:
    print(f"  {NAGL_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")
print(f"Partial charge model: {PARTIAL_CHARGE_METHOD}")
print(f"Run OpenFF parameterization: {RUN_OPENFF_PARAMETERIZATION}")
print(f"Run vacuum collapse: {RUN_VACUUM_COLLAPSE}")
print(f"Run periodic NPT after collapse: {RUN_PERIODIC_NPT}")

## 5. Convert RDKit Molecules to OpenFF Molecules

In [ ]:
def transfer_rdkit_metadata_to_openff(rdkit_mol: Chem.Mol, off_mol: Molecule) -> None:
    """Copy SDF atom-property metadata into OpenFF atom metadata."""
    for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
        props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
        residue_name = str(props.get("residue_name", props.get("mupt_residue_label", "UNK")))
        residue_number = str(props.get("residue_id", props.get("mupt_residue_index", "1")))
        chain_id = str(props.get("chain_id", "A"))

        off_atom.metadata.update({
            "residue_name": residue_name,
            "residue_number": residue_number,
            "chain_id": chain_id,
            "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
        })


off_molecules = []
if OPENFF_AVAILABLE:
    for mol in rdkit_mols:
        off_mol = Molecule.from_rdkit(
            mol,
            allow_undefined_stereo=True,
            hydrogens_are_explicit=True,
        )
        transfer_rdkit_metadata_to_openff(mol, off_mol)
        off_molecules.append(off_mol)
    print(f"Created {len(off_molecules)} OpenFF Molecule object(s)")
    first_atom = off_molecules[0].atom(0)
    print("First OpenFF atom metadata:")
    print(f"  residue_name: {first_atom.metadata.get('residue_name')}")
    print(f"  residue_number: {first_atom.metadata.get('residue_number')}")
    print(f"  chain_id: {first_atom.metadata.get('chain_id')}")
else:
    print("Skipping OpenFF conversion because openff-toolkit is not installed.")


## 6. GNN-Charged Vacuum Parameterization

This cell assigns charges per molecule with the OpenFF GNN model, parameterizes each SDF-loaded molecule with its random-walk coordinates, then combines the molecule interchanges into one vacuum system. This avoids slow AM1-BCC and avoids building an enormous sparse PME box around the initially linear chains.

The important detail is the explicit `NAGLToolkitWrapper` registry. Without it, OpenFF only tries RDKit, AmberTools, and the built-in toolkit wrappers, none of which provide `openff-gnn-am1bcc-1.0.0.pt`. If NAGL is unavailable, the cell skips rather than using AM1-BCC.

In [ ]:
from functools import reduce

interchange = None
mol_interchanges = []

if OPENFF_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField(FORCE_FIELD)
    nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])

    for mol_idx, off_mol in enumerate(off_molecules):
        print(f"Charging and parameterizing molecule {mol_idx + 1}/{len(off_molecules)}")
        off_mol.assign_partial_charges(
            partial_charge_method=PARTIAL_CHARGE_METHOD,
            toolkit_registry=nagl_registry,
        )
        mol_inc = ff.create_interchange(
            off_mol.to_topology(),
            charge_from_molecules=[off_mol],
        )
        mol_inc.box = None
        mol_interchanges.append(mol_inc)

    interchange = reduce(Interchange.combine, mol_interchanges)
    interchange.box = None
    print(f"Combined vacuum interchange with {interchange.topology.n_atoms} atoms")
elif OPENFF_AVAILABLE and not NAGL_AVAILABLE:
    print(
        "OpenFF parameterization skipped because OpenFF NAGL is unavailable. "
        "Install openff-nagl to use openff-gnn-am1bcc-1.0.0.pt. "
        "This tutorial intentionally does not fall back to AM1-BCC because "
        "AM1-BCC is slow for polymer chains."
    )
else:
    print("OpenFF parameterization skipped. Install openff-toolkit and openff-nagl to run it.")

## 7. Vacuum Coordinate Diagnostics

The SDF random-walk coordinates supply the vacuum starting structure. This cell checks for obvious overlaps without constructing an O(N²) distance matrix.

In [ ]:
def minimum_pair_distance_nm(positions_nm: np.ndarray) -> float:
    """Return the nearest-neighbor distance without an O(N^2) matrix."""
    from scipy.spatial import cKDTree

    distances, _ = cKDTree(positions_nm).query(positions_nm, k=2)
    return float(np.min(distances[:, 1]))


if interchange is not None:
    positions_nm = interchange.positions.m_as(off_unit.nanometer)
    min_distance_nm = minimum_pair_distance_nm(positions_nm)
    span_nm = positions_nm.max(axis=0) - positions_nm.min(axis=0)
    print(f"Minimum vacuum starting atom-atom distance: {min_distance_nm:.4f} nm")
    print(f"Initial coordinate span: {span_nm[0]:.2f} x {span_nm[1]:.2f} x {span_nm[2]:.2f} nm")
    if min_distance_nm < 0.005:
        raise ValueError(
            "Detected overlapping or nearly overlapping vacuum coordinates before OpenMM. "
            "Regenerate SDFs from the quickstart notebook with larger chain spacing."
        )
else:
    print("Vacuum coordinate diagnostics skipped because no Interchange was created.")


## 8. Vacuum Collapse and Optional Periodic NPT

This cell first runs vacuum minimization and a short vacuum MD segment so the initially linear polymer conformations can relax without PME grid costs. It then updates the `Interchange` coordinates from that collapsed state. If `RUN_PERIODIC_NPT = True`, it wraps a padded periodic box around the collapsed coordinates and starts a separate NPT simulation from that state.

In [ ]:
vacuum_simulation = None
vacuum_state = None
periodic_simulation = None
periodic_state = None
openmm_dir = SIM_DIR / "OpenMM"
openmm_dir.mkdir(parents=True, exist_ok=True)

def set_periodic_box_around_positions(interchange, padding_nm: float):
    """Translate coordinates into a padded orthorhombic periodic box."""
    positions_nm = interchange.positions.m_as(off_unit.nanometer)
    mins = positions_nm.min(axis=0)
    maxs = positions_nm.max(axis=0)
    shifted_positions = positions_nm - mins + padding_nm
    box_lengths = (maxs - mins) + 2 * padding_nm
    interchange.positions = shifted_positions * off_unit.nanometer
    interchange.box = np.diag(box_lengths) * off_unit.nanometer
    return box_lengths


if interchange is not None and OPENMM_AVAILABLE and RUN_VACUUM_COLLAPSE:
    temperature = 300.0 * omm_unit.kelvin
    pressure = 1.0 * omm_unit.atmosphere
    vacuum_time_step = 0.5 * omm_unit.femtosecond
    periodic_time_step = 1.0 * omm_unit.femtosecond
    friction = 1.0 / omm_unit.picosecond
    report_interval = 25

    print("Creating non-periodic vacuum OpenMM simulation")
    interchange.box = None
    vacuum_integrator = LangevinMiddleIntegrator(temperature, friction, vacuum_time_step)
    vacuum_simulation = interchange.to_openmm_simulation(
        integrator=vacuum_integrator,
        combine_nonbonded_forces=False,
    )

    print("Running vacuum energy minimization...")
    vacuum_simulation.minimizeEnergy()
    if VACUUM_COLLAPSE_STEPS:
        print(f"Running {VACUUM_COLLAPSE_STEPS} vacuum MD steps to collapse chains...")
        vacuum_simulation.step(VACUUM_COLLAPSE_STEPS)
    vacuum_state = vacuum_simulation.context.getState(getEnergy=True, getPositions=True)
    collapsed_positions_nm = vacuum_state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
    interchange.positions = collapsed_positions_nm * off_unit.nanometer
    print(f"Vacuum potential energy: {vacuum_state.getPotentialEnergy()}")

    system_name = "role_aware_saamr"
    vacuum_topology_path = openmm_dir / f"{system_name}_vacuum_topology.pdb"
    vacuum_system_path = openmm_dir / f"{system_name}_vacuum_system.xml"
    vacuum_state_path = openmm_dir / f"{system_name}_vacuum_state.xml"
    with vacuum_topology_path.open("w") as handle:
        PDBFile.writeFile(vacuum_simulation.topology, vacuum_state.getPositions(asNumpy=True), handle)
    vacuum_system_path.write_text(XmlSerializer.serialize(vacuum_simulation.system))
    vacuum_state_path.write_text(XmlSerializer.serialize(vacuum_state))
    print("Serialized vacuum OpenMM components:")
    for path in (vacuum_topology_path, vacuum_system_path, vacuum_state_path):
        print(f"  {path.relative_to(EXAMPLES_ROOT)}")

    if RUN_PERIODIC_NPT:
        box_lengths = set_periodic_box_around_positions(interchange, PERIODIC_PADDING_NM)
        print(
            f"Starting periodic NPT with box lengths: {box_lengths[0]:.2f} x "
            f"{box_lengths[1]:.2f} x {box_lengths[2]:.2f} nm"
        )
        periodic_integrator = LangevinMiddleIntegrator(temperature, friction, periodic_time_step)
        periodic_simulation = interchange.to_openmm_simulation(
            integrator=periodic_integrator,
            combine_nonbonded_forces=False,
            additional_forces=[MonteCarloBarostat(pressure, temperature, 25)],
        )
        print("Running periodic minimization before NPT dynamics...")
        periodic_simulation.minimizeEnergy()
        if PERIODIC_NPT_STEPS:
            dcd_path = openmm_dir / f"{system_name}_periodic_npt_trajectory.dcd"
            state_data_path = openmm_dir / f"{system_name}_periodic_npt_state_data.csv"
            periodic_simulation.reporters.append(DCDReporter(str(dcd_path), report_interval))
            periodic_simulation.reporters.append(
            StateDataReporter(
                str(state_data_path),
                reportInterval=report_interval,
                step=True,
                time=True,
                potentialEnergy=True,
                kineticEnergy=True,
                temperature=True,
                volume=True,
                density=True,
                speed=True,
            )
            )
            print(f"Running {PERIODIC_NPT_STEPS} periodic NPT steps...")
            periodic_simulation.step(PERIODIC_NPT_STEPS)
            print(f"Trajectory saved to: {dcd_path.relative_to(EXAMPLES_ROOT)}")
        periodic_state = periodic_simulation.context.getState(getEnergy=True, getPositions=True)
        periodic_state_path = openmm_dir / f"{system_name}_periodic_npt_state.xml"
        periodic_state_path.write_text(XmlSerializer.serialize(periodic_state))
        print(f"Periodic NPT potential energy: {periodic_state.getPotentialEnergy()}")
else:
    print("Vacuum collapse skipped. Create an Interchange and enable RUN_VACUUM_COLLAPSE to run it.")

## 9. Optional Export Templates

These cells are templates for downstream exporters. They are disabled by default so tutorial execution does not create heavy MD artifacts.

In [ ]:
RUN_GROMACS_EXPORT = False
RUN_LAMMPS_EXPORT = False

if interchange is not None and RUN_GROMACS_EXPORT:
    gromacs_prefix = SIM_DIR / "role_aware_saamr"
    interchange.to_gromacs(str(gromacs_prefix), decimal=5)
    print(f"Wrote GROMACS files with prefix {gromacs_prefix}")
else:
    print("GROMACS export skipped.")

if interchange is not None and RUN_LAMMPS_EXPORT:
    print("LAMMPS export hook goes here once a project-standard exporter is selected.")
else:
    print("LAMMPS export skipped.")

## 10. Summary

This notebook is the downstream half of the workflow. The first notebook builds role-aware MuPT SAAMR systems and writes SDF files; this notebook loads those files, assigns GNN partial charges, parameterizes with OpenFF, and provides the OpenMM execution path used for simulation setup.